# 07 — Expected Loss, IFRS 9 and IRB Grading

Extend the scorecard (PD output) to a euro-denominated expected loss,
IFRS 9 staging, and a five-grade risk classification.

**Explicit assumptions (German Credit):**
- LGD = 45 % (unsecured consumer loan proxy)
- EAD = initial credit amount (approximation, no default-time exposure data)
- CCF = not estimable (no off-balance-sheet commitment)
- Lifetime PD ≈ 12-month PD × duration / 12

This is a teaching demonstrator, not a regulatory IFRS 9 model.

In [3]:
import sys, json, os, sqlite3
sys.path.append("../src")

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

from decision import score_to_pd
from credit_risk import (
    expected_loss, stressed_lgd, ifrs9_stage, provision as ifrs9_provision,
    reason_codes_for_client, adverse_impact_by_group,
)

params = json.load(open("../models/scaling_params.json"))
OFFSET, FACTOR, INTERCEPT = params["Offset"], params["Factor"], params["intercept"]
features = json.load(open("../models/selected_features.json"))

te = pd.read_parquet("../data/processed/test_scores_with_target.parquet")
tr = pd.read_parquet("../data/processed/train_scores_with_target.parquet")
X_train = pd.read_parquet("../data/processed/X_train_woe.parquet")
X_test = pd.read_parquet("../data/processed/X_test_woe.parquet")

# Clients table already has renamed columns from data_loader.py
con = sqlite3.connect("../data/processed/german_credit.db")
clients = pd.read_sql_query("SELECT credit_amount, duration, personal_status_sex FROM clients", con)
con.close()

# Align on test indices — never take the last 150 rows
EAD = clients.iloc[te.index]["credit_amount"].values.astype(float)
DURATION = clients.iloc[te.index]["duration"].values.astype(float)
SEX = clients.iloc[te.index]["personal_status_sex"].values

pd_te = np.asarray(score_to_pd(te["score"].values, OFFSET, FACTOR, INTERCEPT)).flatten()
y_te = te["default"].values

print(f"Test set: {len(te)} clients | defaults: {int(y_te.sum())}")
print(f"EAD min/mean/max: {EAD.min():.0f} / {EAD.mean():.0f} / {EAD.max():.0f}")

Test set: 150 clients | defaults: 45
EAD min/mean/max: 250 / 3183 / 18424


## 1. From PD to Expected Loss

Formula : `EL = PD × LGD × EAD`.

We also compute `EL_rate = PD × LGD`, which is independent of the EAD
approximation and therefore more robust to scrutiny.

In [4]:
LGD = 0.45
EL = expected_loss(pd_te, LGD, EAD)
EL_RATE = pd_te * LGD

df = pd.DataFrame({
    "score": te["score"].values,
    "pd": pd_te,
    "ead": EAD,
    "el": EL,
    "el_rate": EL_RATE,
    "default": y_te,
})
df["decile"] = pd.qcut(df["pd"], 10, labels=False) + 1

table = df.groupby("decile").agg(
    n=("default", "size"),
    pd_moy=("pd", "mean"),
    el_moy=("el", "mean"),
    taux_defaut=("default", "mean"),
)
print(table.to_string())
print(f"\nTotal portfolio EL: {EL.sum():,.2f} EUR")

         n    pd_moy       el_moy  taux_defaut
decile                                        
1       15  0.019736    18.451644     0.000000
2       15  0.047479    51.094896     0.133333
3       15  0.076944    85.627482     0.133333
4       15  0.120052   102.497781     0.133333
5       15  0.172574   280.574934     0.200000
6       15  0.235144   222.298505     0.066667
7       15  0.323014   582.760375     0.466667
8       15  0.425650   686.316315     0.466667
9       15  0.580235   698.232133     0.600000
10      15  0.783779  2466.698564     0.800000

Total portfolio EL: 77,918.29 EUR


## 2. IRB Grading via Clustering

Fit KMeans (k=5) on the 700-row training set, apply to test, then check
that default rates rise with mean PD. If not monotonic, keep the existing
score bands as the reference instead.

In [5]:
# Fit on train only — fitting on test would leak information
km = KMeans(n_clusters=5, random_state=42, n_init=10)
km.fit(X_train[features])

te["cluster"] = km.predict(X_test[features])

grading = te.assign(pd=pd_te).groupby("cluster").agg(
    n=("default", "size"),
    pd_moy=("pd", "mean"),
    taux_defaut=("default", "mean"),
).sort_values("pd_moy")

grading["grade"] = range(1, len(grading) + 1)
print(grading.to_string())

monotone = grading["taux_defaut"].is_monotonic_increasing
print(f"\nMonotonic default rate by grade: {monotone}")
if not monotone:
    print("Non-monotonic grading — do not claim an IRB rating system.")

          n    pd_moy  taux_defaut  grade
cluster                                  
3        17  0.095280     0.117647      1
0        13  0.115675     0.153846      2
2        51  0.133984     0.176471      3
4        17  0.265306     0.352941      4
1        52  0.525041     0.500000      5

Monotonic default rate by grade: True


## 3. IFRS 9 Staging and Provisions

Stage 1: normal risk → 12-month EL
Stage 2: significant deterioration (SICR) → lifetime EL
Stage 3: observed default → lifetime EL

Stress: PD × 1.2 and LGD × 1.25 to simulate a downturn.
This complements the existing model-robustness stress tests, which target
the model rather than the provisions.

In [9]:
from credit_risk import (
    expected_loss, stressed_lgd, ifrs9_stage,
    provision as ifrs9_provision, top_reason_codes,
)

# Lifetime PD proxy — no maturity ladder in the dataset
PD_LIFETIME = np.clip(pd_te * DURATION / 12.0, 0, 1)
stage = ifrs9_stage(pd_te, PD_LIFETIME, sicr_threshold=2.0, default_flag=y_te)

prov = ifrs9_provision(pd_te, PD_LIFETIME, LGD, EAD, stage)
prov_stress = ifrs9_provision(
    np.clip(pd_te * 1.2, 0, 1),
    np.clip(PD_LIFETIME * 1.2, 0, 1),
    stressed_lgd(LGD, factor=1.25),
    EAD,
    stage,
)

print(pd.DataFrame({"stage": stage, "provision": prov})
      .groupby("stage").agg(n=("provision", "size"),
                            total=("provision", "sum")).to_string())
print(f"\nTotal provision: {prov.sum():,.2f} EUR")
print(f"Under stress:    {prov_stress.sum():,.2f} EUR "
      f"(+{(prov_stress.sum()/prov.sum()-1):.1%})")

        n         total
stage                  
1      91  19058.474129
2      14  16965.290494
3      45  72648.494640

Total provision: 108,672.26 EUR
Under stress:    145,417.33 EUR (+33.8%)


## 4. Adverse-Action Reason Codes

Refusals must be explainable. For each rejected client, list the three
variables that removed the most points according to `scorecard.csv`.

In [ ]:
scorecard = pd.read_csv("../models/scorecard.csv")

SEUIL = 1049.11  # score threshold — accept if score > SEUIL
refused = np.where(te["score"].values <= SEUIL)[0][:5]
print(f"Refused clients shown: {len(refused)}")

for idx in refused:
    codes = reason_codes_for_client(X_test.iloc[idx][features], features, scorecard, n=3)
    print(f"Client {te.index[idx]} | score {te['score'].iloc[idx]:.1f} | "
          f"reasons: {[c[0] for c in codes]}")

0.0036504644914531104 0.9333426910198817
926.5196762270352 1164.5196762270355


## 5. Adverse-Impact Analysis by Sex

`personal_status_sex` is a sensitive attribute kept in the model. This section
checks approval rate, default rate and mean cost per group. If the gap is large,
the attribute should be removed or challenged.

In [ ]:
approved = te["score"].values > SEUIL
impact = adverse_impact_by_group(SEX, y_te, approved)
print(impact.to_string())
print(f"\nMax mean-cost gap: {impact['mean_cost'].max() - impact['mean_cost'].min():.3f} units")

## 6. Save Artifacts


In [15]:
os.makedirs("../models", exist_ok=True)

with open("../models/irb_ifrs9.json", "w") as f:
    json.dump({
        "lgd_assumption": LGD,
        "ead_proxy": "credit_amount (Attribute5)",
        "ccf": "not estimable - no off-balance-sheet commitment",
        "pd_lifetime_rule": "PD_12m * duration / 12",
        "sicr_rule": "PD_lifetime > 2 * PD_12m",
        "el_total": round(float(EL.sum()), 2),
        "provision_total": round(float(prov.sum()), 2),
        "provision_stressed": round(float(prov_stress.sum()), 2),
        "grading_monotone": bool(monotone),
    }, f, indent=2)

grading.to_csv("../models/irb_grading.csv")
print("Saved models/irb_ifrs9.json and models/irb_grading.csv")

Saved models/irb_ifrs9.json and models/irb_grading.csv


108672.25926428975 145417.33345842277
